**Dataset:** COCA news sample, `text_news.txt`

**Tasks**

1. Produce the normalized text: cleaned and lemmatized.
2. Find words similar to *time* with $BM25_{norm} \odot IDF$ and cosine
   similarity.
3. Find words similar to *time* with relative frequency and cosine similarity.
4. Find the best result by testing other configurations: dot product,
   Euclidean distance, no lemmatization, and a frequency filter.


pre-processing -> vectorization -> similarity

raw text -> normalized text -> vocabulary -> contexts (bags of words) ->
vectors -> similarity with *time*

In [1]:
# Libraries:
# - re: delete every character that is not a letter.
# - Counter: count how many times each word appears.
# - nltk: tokenizer and English stopword list.
# - spacy: lemmatizer (it uses the part of speech of each word).
# - CountVectorizer: turns each context (a list of words) into a row of counts.
# - numpy: vector arithmetic for the formulas.
# - pandas: display the result tables.
import re
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

# Data files used by the NLTK tokenizer and stopword list.
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# Folder for the normalized text and the full result tables.
output_folder = Path("results/generated")
output_folder.mkdir(exist_ok=True)

# The word whose similar words we want to find.
target = "time"

## Task 1: normalized text (cleaned and lemmatized)

Normalization:

Lowercase: *Time* -> *time* 

Tokenize: Split the text into words 

Clean: Keep only letters

Stopwords: Delete *the*, *of*, *and*... 

Lemmatize: *times*, *timed* -> *time* 

In [2]:
source_path = Path("data/text_news.txt")

# The course dataset is kept outside Git and placed in data/ locally.
# Its bytes are ASCII, so ASCII, Latin-1, and UTF-8 decode it equally.
with open(source_path, encoding="ascii") as file:
    raw_text = file.read()

print("Characters in the raw text:", len(raw_text))


Characters in the raw text: 7018110


In [3]:
# Step 1: lowercase, then tokenize with NLTK.
raw_tokens = nltk.word_tokenize(raw_text.lower())

# Step 2: keep only the letters of each token. Tokens with no letters
# (numbers, punctuation) disappear.
clean_tokens = []
for token in raw_tokens:
    letters = re.sub(r"[^a-z]", "", token)
    if letters:
        clean_tokens.append(letters)

# Step 3: delete stopwords.
stopword_set = set(stopwords.words("english"))
tokens_without_stopwords = [t for t in clean_tokens if t not in stopword_set]

# COCA marks paragraphs with <p> and headings with <h>. After cleaning, only
# the letters "p" and "h" remain. They are markup, not words, so they go.
markup_leftovers = {"p", "h"}
tokens_without_stopwords = [
    t for t in tokens_without_stopwords if t not in markup_leftovers
]

print("Raw tokens:              ", len(raw_tokens))
print("Clean tokens:            ", len(clean_tokens))
print("Tokens without stopwords:", len(tokens_without_stopwords))

Raw tokens:               1448073
Clean tokens:             1117129
Tokens without stopwords: 622148


In [4]:
# Step 4: lemmatize with spaCy. The parser and the named-entity recognizer
# are not needed for lemmas, so they are disabled to save time.
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# spaCy limits the length of one input, so the tokens are sent in chunks of
# 5,000 words. nlp.pipe processes the chunks one after another.
chunk_size = 5000
chunks = [
    " ".join(tokens_without_stopwords[i:i + chunk_size])
    for i in range(0, len(tokens_without_stopwords), chunk_size)
]

normalized_tokens = []
noun_counts = Counter()  # how many times each lemma was tagged as a noun
for doc in nlp.pipe(chunks):
    for token in doc:
        lemma = token.lemma_.lower()
        # Keep only lemmas made of letters (skip empty or symbol lemmas).
        # The stopword list is applied again: lemmatization can turn a
        # remaining token into a stopword ("nt" -> "not", "us" -> "we").
        if (lemma.isalpha() and lemma not in markup_leftovers
                and lemma not in stopword_set):
            normalized_tokens.append(lemma)
            # spaCy also tags the part of speech. It is kept so that task 4
            # can rank only nouns, because "time" is a noun.
            if token.pos_ == "NOUN":
                noun_counts[lemma] += 1

# spaCy tokenizes the joined text again and splits a few words written
# without an apostrophe ("id" -> "i" + "d"), so the token count can
# change slightly in this step.

# Save the normalized text (task 1), 20 words per line so it is readable.
normalized_path = output_folder / "normalized-text.txt"
with open(normalized_path, "w", encoding="utf-8") as file:
    for i in range(0, len(normalized_tokens), 20):
        file.write(" ".join(normalized_tokens[i:i + 20]) + "\n")

pd.DataFrame(
    [
        ("Raw tokens", len(raw_tokens), len(set(raw_tokens))),
        ("Clean tokens", len(clean_tokens), len(set(clean_tokens))),
        ("Without stopwords", len(tokens_without_stopwords),
         len(set(tokens_without_stopwords))),
        ("Lemmatized (normalized)", len(normalized_tokens),
         len(set(normalized_tokens))),
    ],
    columns=["step", "tokens", "vocabulary size"],
)

,step,tokens,vocabulary size
0,Raw tokens,1448073,58341
1,Clean tokens,1117129,51133
2,Without stopwords,622148,50993
3,Lemmatized (normalized),615788,43313


In [5]:
print("Saved:", normalized_path)

# The lemmatizer joins the forms of "time" into one word.
word_counts = Counter(normalized_tokens)
print("\nBefore lemmatization: 'time' =", tokens_without_stopwords.count("time"),
      "| 'times' =", tokens_without_stopwords.count("times"))
print("After lemmatization:  'time' =", word_counts[target],
      "| 'times' =", word_counts["times"])

# Not every "times" becomes "time": spaCy tags "times" in names such as
# "The New York Times" as a proper noun and keeps it unchanged. This is
# correct, because the newspaper is not the word "time".

Saved: results/generated/normalized-text.txt

Before lemmatization: 'time' = 1577 | 'times' = 499
After lemmatization:  'time' = 1955 | 'times' = 137


## Setup for tasks 2-4: vocabulary and contexts (bags of words)

The **vocabulary** is the list of unique normalized words

The **context** of a word is the list of words around each of its
occurrences, with a window of 8 (4 words to the left and 4 to the right).
Each context is treated as a **document** $d$, a bag of words: order does not
matter, but repetitions do. Its length is the number of words in it.

Because stopwords were removed, 4 normalized words can cover more than 4
original words, and a window can cross from one article into the next.

The contexts are collected in **one pass** over the text: each position adds
its neighbors to the context of the word at that position.

In [6]:
window = 4  # 4 words to the left + 4 to the right = window of 8

contexts = {}
for position, word in enumerate(normalized_tokens):
    left = normalized_tokens[max(0, position - window):position]
    right = normalized_tokens[position + 1:position + window + 1]
    contexts.setdefault(word, []).extend(left + right)

# The vocabulary is the list of words that have a context, every
# unique normalized word. Its order defines the rows and the columns
# of the count matrix
vocabulary = list(contexts)
M = len(vocabulary)
t = vocabulary.index(target)  # row (and column) number of "time"

print("Vocabulary size M:", M)


Vocabulary size M: 43313


### Raw frequency: $c(w_i, d)$

`CountVectorizer` turns each context into a row of counts. Row $j$ is the
context of word $w_j$; column $i$ counts how many times $w_i$ appears in it:
$c(w_i, d_j)$. Passing `vocabulary=vocabulary` makes the columns use the same
word order as the rows, so column $i$ and row $i$ are the same word.

The raw frequency is **biased**: frequent words have long contexts and large
counts, and very common words are in almost every context. The weighting
schemes below correct this.

In [7]:
# Each context becomes one string because CountVectorizer expects text.
context_strings = [" ".join(contexts[word]) for word in vocabulary]

# token_pattern keeps every word (the default drops one-letter words) and
# lowercase=False because the text is already lowercase.
vectorizer = CountVectorizer(
    vocabulary=vocabulary, token_pattern=r"(?u)\b\w+\b", lowercase=False
)
counts = vectorizer.fit_transform(context_strings).astype(float)

print("Count matrix (documents x words):", counts.shape)
print("Nonzero values:", counts.nnz)

# Raw counts of a few context words in the context of "time".
sample = ["first", "last", "long", "day", "year", "spend"]
pd.DataFrame(
    {"c(w, d_time)": [counts[t, vocabulary.index(w)] for w in sample]},
    index=sample,
)

Count matrix (documents x words): (43313, 43313)
Nonzero values: 3231361


,"c(w, d_time)"
first,154.0
last,46.0
long,78.0
day,48.0
year,119.0
spend,61.0


### Setup: quantities shared by the formulas

| Symbol | Formula | Meaning |
| --- | --- | --- |
| $len(d_j)$ | $\sum_i c(w_i, d_j)$ | Number of words in context $j$ (sum of the row) |
| $avdl$ | $\frac{1}{M}\sum_j len(d_j)$ | Average context length |
| $df(w_i)$ | number of $d_j$ with $c(w_i, d_j) > 0$ | Number of contexts that contain $w_i$ |
| $IDF(w_i)$ | $\log \frac{M + 1}{df(w_i)}$ | High for rare context words, low for common ones |


IDF is needed because a word that is in almost every context (such as *say*)
does not tell us anything about meaning. The $+1$ keeps the value positive
even for a word that is in every context.

In [8]:
# len(d): sum of each row of the count matrix. A sparse sum returns a
# 2-D matrix (M x 1); np.asarray + .ravel() turn it into a flat array.
doc_lengths = np.asarray(counts.sum(axis=1)).ravel()

# avdl: average context length.
avdl = doc_lengths.sum() / M

# df(w): number of rows (contexts) where the column of w is not zero.
# (counts > 0) gives True/False; True counts as 1 in the sum.
df = np.asarray((counts > 0).sum(axis=0)).ravel()

# IDF(w) = log((M + 1) / df(w)).
idf = np.log((M + 1) / df)

print("len(d_time):", int(doc_lengths[t]))
print("avdl:       ", round(avdl, 1))
print("IDF range:  ", round(idf.min(), 2), "to", round(idf.max(), 2))

# Word indices from lowest to highest IDF, only for the example table.
order = np.argsort(idf)
pd.DataFrame({
    "lowest IDF": [vocabulary[i] for i in order[:8]],
    "IDF": idf[order[:8]].round(2),
    "highest IDF": [vocabulary[i] for i in order[-8:]],
    "IDF ": idf[order[-8:]].round(2),
})

len(d_time): 15640
avdl:        113.7
IDF range:   1.25 to 9.29


,lowest IDF,IDF,highest IDF,IDF
0,say,1.25,thailandvalue,9.07
1,one,1.85,filth,9.07
2,year,1.87,spearman,9.29
3,make,1.97,etherton,9.29
4,new,2.03,chery,9.29
5,get,2.08,maldive,9.29
6,like,2.08,moten,9.29
7,go,2.08,capon,9.29


## Task 2: $BM25_{norm} \odot IDF$ and cosine similarity

**Step 1: BM25.** Each raw count becomes

$$BM25(c) = \frac{(k + 1)\, c(w_i, d)}{c(w_i, d) + k\left(1 - b + b\,\frac{len(d)}{avdl}\right)}$$

- **Saturation:** the value grows quickly for small counts and then flattens
  (it can never exceed $k + 1$), so one very frequent context word cannot
  dominate the vector.
- **Length normalization:** $\frac{len(d)}{avdl}$ makes counts in a long
  context (a frequent word) count less than the same counts in a short one.
- $k \geq 0$ controls the saturation and $b \in [0, 1]$ the length
  normalization. The common values $k = 1.2$ and $b = 0.75$ are used.

**Step 2: normalize.** Each BM25 value is divided by the sum of all the values
in its vector, so every vector sums to 1 (like probabilities):

$$BM25_{norm}(w_i, d) = \frac{BM25(w_i, d)}{\sum_{l=1}^{M} BM25(w_l, d)}$$

**Step 3: IDF.** The Hadamard (element-by-element) product with the IDF
vector gives common context words a low weight:

$$\vec{X} = BM25_{norm} \odot IDF \qquad x_i = BM25_{norm}(w_i, d) \cdot IDF(w_i)$$

**Step 4: cosine similarity** with the vector of *time*:

$$\cos(\vec{X}, \vec{Y}) = \frac{\vec{X} \cdot \vec{Y}}{\|\vec{X}\|_2 \, \|\vec{Y}\|_2}
= \frac{\sum_i x_i y_i}{\sqrt{\sum_i x_i^2}\,\sqrt{\sum_i y_i^2}}$$

Cosine compares the direction of the vectors (which context words matter
most), not their length.

In [9]:
k = 1.2
b = 0.75

# The matrix is sparse: it only stores the nonzero counts. In COO format,
# .data holds the counts and .row the context (document) of each count,
# so the formula can be applied to every count at once.
bm25 = counts.tocoo(copy=True)
c = bm25.data
len_d = doc_lengths[bm25.row]

# Step 1: BM25(c) = (k + 1) c / (c + k (1 - b + b len(d) / avdl)).
# Zero counts stay zero, so only the nonzero values need the formula.
bm25.data = (k + 1) * c / (c + k * (1 - b + b * len_d / avdl))
# Back to CSR format, which is fast for row operations (row sums,
# selecting the row of "time").
bm25 = bm25.tocsr()

# Step 2: divide each row by the sum of its BM25 values, so it sums to 1.
bm25_row_sums = np.asarray(bm25.sum(axis=1)).ravel()
# [:, np.newaxis] turns the M sums into a column (M x 1), so .multiply
# divides each ROW by its own sum. .multiply is element by element,
# not matrix multiplication.
bm25_norm = bm25.multiply(1 / bm25_row_sums[:, np.newaxis]).tocsr()

# Step 3: Hadamard product with IDF. Column i of every row is multiplied
# by IDF(w_i). idf[np.newaxis, :] is a row (1 x M), so it is applied to
# every row of the matrix.
X = bm25_norm.multiply(idf[np.newaxis, :]).tocsr()

# Example: how BM25 flattens the counts in the context of "time".
# .toarray().ravel(): the row of "time" as a normal flat array.
row = counts[t].toarray().ravel()
# Indices of the 6 largest counts ([::-1] reverses to descending order).
largest = np.argsort(row)[::-1][:6]
pd.DataFrame({
    "context word": [vocabulary[i] for i in largest],
    "raw count": row[largest],
    "BM25": bm25[t].toarray().ravel()[largest].round(3),
    "BM25_norm": bm25_norm[t].toarray().ravel()[largest].round(5),
    "IDF": idf[largest].round(2),
    "X = BM25_norm * IDF": X[t].toarray().ravel()[largest].round(5),
})

,context word,raw count,BM25,BM25_norm,IDF,X = BM25_norm * IDF
0,say,286.0,1.534,0.00626,1.25,0.00783
1,first,154.0,1.218,0.00497,2.28,0.01133
2,year,119.0,1.077,0.00440,1.87,0.00822
3,get,112.0,1.044,0.00426,2.08,0.00886
4,go,110.0,1.034,0.00422,2.08,0.00878
5,take,106.0,1.014,0.00414,2.11,0.00875


In [10]:
print("BM25_norm of 'time' sums to:", bm25_norm[t].sum().round(6))

# Step 4: cosine similarity between X(time) and the vector of every word.
# Numerator: dot product sum_i x_i y_i for all words at once
# (matrix times the vector of "time"). @ is matrix multiplication: each
# row of X times the column X[t].T gives one dot product per word.
dot_products = (X @ X[t].T).toarray().ravel()

# Denominator: Euclidean norms sqrt(sum_i x_i^2) of every row.
# X.multiply(X) squares each value; the row sum adds the squares.
norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())

cosine_bm25 = dot_products / (norms * norms[t])

# Table with every vocabulary word, from most to least similar.
# "time" itself comes first with similarity 1. sort_values orders the
# rows from high to low; reset_index renumbers them 0, 1, 2...
table_bm25 = (
    pd.DataFrame({
        "word": vocabulary,
        "frequency": [word_counts[w] for w in vocabulary],
        "cosine similarity": cosine_bm25,
    })
    .sort_values("cosine similarity", ascending=False)
    .reset_index(drop=True)
)
table_bm25.to_csv(output_folder / "task2-bm25-idf-cosine.csv", index=False)

print("Rows in the table:", len(table_bm25))
table_bm25.head(20)

BM25_norm of 'time' sums to: 1.0
Rows in the table: 43313


,word,frequency,cosine similarity
0,time,1955,1.000000
1,say,8606,0.782245
2,one,2940,0.771049
3,go,2388,0.767833
4,get,2356,0.767151
5,come,1470,0.735453
6,would,2254,0.733815
7,make,2405,0.722585
8,people,1979,0.721207
9,could,1435,0.711915


## Task 3: relative frequency and cosine similarity

The relative frequency divides each count by the length of its context:

$$x_i = \frac{c(w_i, d)}{len(d)}$$

Each value is the probability that a word picked at random from the context
is $w_i$, and every vector sums to 1. It corrects the context length, but it
does not saturate high counts and does not use IDF.

In [11]:
# x_i = c(w_i, d) / len(d): divide each row by its length. Same
# broadcasting as in task 2: [:, np.newaxis] makes len(d) a column, so
# each row is divided by its own length.
relative = counts.multiply(1 / doc_lengths[:, np.newaxis]).tocsr()
print("Relative frequencies of 'time' sum to:", relative[t].sum().round(6))

# Same cosine formula as in task 2: dot products divided by the norms.
dot_products = (relative @ relative[t].T).toarray().ravel()
norms = np.sqrt(np.asarray(relative.multiply(relative).sum(axis=1)).ravel())
cosine_relative = dot_products / (norms * norms[t])

# Table with every vocabulary word, from most to least similar.
table_relative = (
    pd.DataFrame({
        "word": vocabulary,
        "frequency": [word_counts[w] for w in vocabulary],
        "cosine similarity": cosine_relative,
    })
    .sort_values("cosine similarity", ascending=False)
    .reset_index(drop=True)
)
table_relative.to_csv(
    output_folder / "task3-relative-frequency-cosine.csv", index=False
)

print("Rows in the table:", len(table_relative))
table_relative.head(20)

Relative frequencies of 'time' sum to: 1.0


Rows in the table: 43313


,word,frequency,cosine similarity
0,time,1955,1.000000
1,get,2356,0.894912
2,come,1470,0.893032
3,go,2388,0.887486
4,one,2940,0.884347
5,could,1435,0.869030
6,work,1465,0.865259
7,still,953,0.861979
8,make,2405,0.861388
9,say,8606,0.856511


## Task 4: other configurations

### 4.1 Vectors without lemmatization

The same steps as the setup and task 2, but on the tokens **before**
lemmatization (cleaned, without stopwords). Here *time* and *times* are
different words with separate vectors.

In [12]:
# Contexts (window of 8) on the tokens before lemmatization.
contexts_nl = {}
for position, word in enumerate(tokens_without_stopwords):
    left = tokens_without_stopwords[max(0, position - window):position]
    right = tokens_without_stopwords[position + 1:position + window + 1]
    contexts_nl.setdefault(word, []).extend(left + right)

vocabulary_nl = list(contexts_nl)
M_nl = len(vocabulary_nl)
# Frequency of each word before lemmatization, used by the filters.
word_counts_nl = Counter(tokens_without_stopwords)

# Raw counts c(w_i, d).
counts_nl = CountVectorizer(
    vocabulary=vocabulary_nl, token_pattern=r"(?u)\b\w+\b", lowercase=False
).fit_transform([" ".join(contexts_nl[w]) for w in vocabulary_nl]).astype(float)

# len(d), avdl, df, and IDF.
doc_lengths_nl = np.asarray(counts_nl.sum(axis=1)).ravel()
avdl_nl = doc_lengths_nl.sum() / M_nl
df_nl = np.asarray((counts_nl > 0).sum(axis=0)).ravel()
idf_nl = np.log((M_nl + 1) / df_nl)

# Same BM25 formula as in task 2, on the new counts (c and len_d are
# reused as names for the new values).
bm25_nl = counts_nl.tocoo(copy=True)
c = bm25_nl.data
len_d = doc_lengths_nl[bm25_nl.row]
bm25_nl.data = (k + 1) * c / (c + k * (1 - b + b * len_d / avdl_nl))
bm25_nl = bm25_nl.tocsr()
bm25_nl_sums = np.asarray(bm25_nl.sum(axis=1)).ravel()
X_nl = (
    bm25_nl.multiply(1 / bm25_nl_sums[:, np.newaxis])
    # Steps 2 and 3 in one expression: divide each row by its BM25
    # sum, then multiply every row by the IDF vector.
    .multiply(idf_nl[np.newaxis, :])
    .tocsr()
)

print("Vocabulary without lemmatization:", M_nl, "(with lemmatization:", M, ")")

Vocabulary without lemmatization: 50993 (with lemmatization: 43313 )


### 4.2 A narrower window

With a window of 8, the context mixes the words right next to *time* with
words that are only in the same sentence (topic words). A window of 2
(1 word to the left and 1 to the right) keeps only the closest neighbors,
such as *first*, *long*, or *spend*. Words that fill the same slot in a
phrase (*the first time*, *the first day*) then get similar vectors. The same
BM25 steps are repeated with `window = 1`.

In [13]:
window_narrow = 1  # 1 word to the left + 1 to the right = window of 2

contexts_w2 = {}
for position, word in enumerate(normalized_tokens):
    left = normalized_tokens[max(0, position - window_narrow):position]
    right = normalized_tokens[position + 1:position + window_narrow + 1]
    contexts_w2.setdefault(word, []).extend(left + right)

# Same vocabulary and order as before, so word counts and nouns still apply.
counts_w2 = CountVectorizer(
    vocabulary=vocabulary, token_pattern=r"(?u)\b\w+\b", lowercase=False
).fit_transform([" ".join(contexts_w2[w]) for w in vocabulary]).astype(float)

# len(d), avdl, df, and IDF for the new contexts.
doc_lengths_w2 = np.asarray(counts_w2.sum(axis=1)).ravel()
avdl_w2 = doc_lengths_w2.sum() / M
df_w2 = np.asarray((counts_w2 > 0).sum(axis=0)).ravel()
idf_w2 = np.log((M + 1) / df_w2)

# Same BM25 formula as in task 2, on the new counts (c and len_d are
# reused as names for the new values).
bm25_w2 = counts_w2.tocoo(copy=True)
c = bm25_w2.data
len_d = doc_lengths_w2[bm25_w2.row]
bm25_w2.data = (k + 1) * c / (c + k * (1 - b + b * len_d / avdl_w2))
bm25_w2 = bm25_w2.tocsr()
bm25_w2_sums = np.asarray(bm25_w2.sum(axis=1)).ravel()
X_w2 = (
    bm25_w2.multiply(1 / bm25_w2_sums[:, np.newaxis])
    # Steps 2 and 3 in one expression: divide each row by its BM25
    # sum, then multiply every row by the IDF vector.
    .multiply(idf_w2[np.newaxis, :])
    .tocsr()
)

print("avdl with window 2:", round(avdl_w2, 1), "(window 8:", round(avdl, 1), ")")

avdl with window 2: 28.4 (window 8: 113.7 )


### 4.3 Similarity measures

Besides cosine, two more measures are tested:

- **Dot product:** $\vec{X} \cdot \vec{Y} = \sum_i x_i y_i$. It is the cosine
  without dividing by the lengths, so long vectors get high scores.
- **Euclidean distance:** $\|\vec{X} - \vec{Y}\|_2 = \sqrt{\sum_i (x_i - y_i)^2}$.
  0 means identical, so it is ordered from **smallest** to largest. To avoid
  subtracting 43,000 dense vectors, the sum is expanded:
  $\sum_i (x_i - y_i)^2 = \sum_i x_i^2 + \sum_i y_i^2 - 2\sum_i x_i y_i$,
  which gives exactly the same value.
  Euclidean distance gives the same order as cosine only when every vector
  has length 1 (L2 norm). Our vectors sum to 1 (L1), which is a different
  normalization, so the two orders are close but not identical.

### 4.4 Filters on the ranked words

- **Frequency >= 20.** A word seen only once has a context of just 8 words,
  too little evidence for a reliable vector, so it can match *time* by
  chance.
- **Nouns only.** *time* is a noun, and the words we expect (*day*, *year*,
  *week*) are nouns too. Frequent verbs such as *say* or *go* appear in the
  same broad contexts, so they get high scores without being similar in
  meaning. A word counts as a noun if spaCy tagged it as a noun in at least
  half of its occurrences.

The filters only choose which words are ranked. The vectors and the scores
still use the complete vocabulary.

### 4.5 How "best" is judged

There is no answer key, so each configuration is scored by how many of its
top 20 words are clearly time-related (a hand-made list: *day*, *year*,
*week*, *moment*...). The list is only a simple, transparent check; the top
words are also shown so they can be judged directly.

In [14]:
# Words that are clearly related to "time" (singular and plural, so the
# version without lemmatization is judged fairly).
time_words = {
    "day", "year", "week", "month", "hour", "minute", "moment", "period",
    "season", "night", "morning", "afternoon", "evening", "weekend",
    "decade", "century", "era", "age", "date", "today", "yesterday",
    "tomorrow", "now", "later", "ago", "past", "future", "history", "summer",
    "winter", "schedule", "deadline", "times", "days", "years", "weeks",
    "months", "hours", "minutes", "moments", "periods", "seasons", "nights",
    "decades", "centuries", "dates",
}

# A lemma is a noun if spaCy tagged it as a noun in at least half of its
# occurrences ("time" is tagged as a noun almost always).
nouns = {w for w in vocabulary if noun_counts[w] >= 0.5 * word_counts[w]}
print("'time' is a noun:", target in nouns, "| nouns in the vocabulary:", len(nouns))

# Each configuration: name, vectors, vocabulary, word counts, measure,
# minimum frequency to be ranked, and whether only nouns are ranked.
# (name, vectors, vocabulary, word counts, measure, min frequency, nouns only)
configurations = [
    ("BM25_norm * IDF + cosine (task 2)", X, vocabulary, word_counts, "cosine", 1, False),
    ("Relative frequency + cosine (task 3)", relative, vocabulary, word_counts, "cosine", 1, False),
    ("BM25_norm * IDF + dot product", X, vocabulary, word_counts, "dot", 1, False),
    ("BM25_norm * IDF + Euclidean", X, vocabulary, word_counts, "euclidean", 1, False),
    ("Relative frequency + dot product", relative, vocabulary, word_counts, "dot", 1, False),
    ("Relative frequency + Euclidean", relative, vocabulary, word_counts, "euclidean", 1, False),
    ("No lemmatization: BM25_norm * IDF + cosine", X_nl, vocabulary_nl, word_counts_nl, "cosine", 1, False),
    ("BM25_norm * IDF + cosine, frequency >= 20", X, vocabulary, word_counts, "cosine", 20, False),
    ("BM25_norm * IDF + cosine, frequency >= 20, nouns", X, vocabulary, word_counts, "cosine", 20, True),
    ("Relative frequency + cosine, frequency >= 20, nouns", relative, vocabulary, word_counts, "cosine", 20, True),
    ("Window 2: BM25_norm * IDF + cosine, frequency >= 20", X_w2, vocabulary, word_counts, "cosine", 20, False),
    ("Window 2: BM25_norm * IDF + cosine, frequency >= 20, nouns", X_w2, vocabulary, word_counts, "cosine", 20, True),
    ("BM25_norm * IDF + dot product, frequency >= 20", X, vocabulary, word_counts, "dot", 20, False),
]

summary = []
# enumerate numbers the configurations from 1; each tuple is unpacked
# into 7 named variables.
for number, (name, vectors, vocab, freq, measure, min_count, nouns_only) in enumerate(
    configurations, start=1
):
    # Row of "time". It differs without lemmatization (another vocabulary).
    i_time = vocab.index(target)

    # sum_i x_i y_i between "time" and every word.
    dot_products = (vectors @ vectors[i_time].T).toarray().ravel()
    # sum_i x_i^2 for every word.
    squared_norms = np.asarray(vectors.multiply(vectors).sum(axis=1)).ravel()

    if measure == "cosine":
        scores = dot_products / (
            np.sqrt(squared_norms) * np.sqrt(squared_norms[i_time])
        )
        column, most_similar_first = "cosine similarity", True
    elif measure == "dot":
        scores = dot_products
        column, most_similar_first = "dot product", True
    else:
        # sqrt(sum x^2 + sum y^2 - 2 sum x y); max(0, .) removes tiny
        # negative values caused by floating-point rounding.
        scores = np.sqrt(np.maximum(
            squared_norms + squared_norms[i_time] - 2 * dot_products, 0
        ))
        column, most_similar_first = "Euclidean distance", False

    table = pd.DataFrame({
        "word": vocab,
        "frequency": [freq[w] for w in vocab],
        column: scores,
    })
    # Keep only the rows that pass the filters (boolean masks).
    table = table[table["frequency"] >= min_count]
    if nouns_only:
        table = table[table["word"].isin(nouns)]
    # Similarities are sorted high to low; distances low to high (0 means
    # identical).
    table = table.sort_values(
        column, ascending=not most_similar_first
    ).reset_index(drop=True)
    # {number:02d} pads with a zero: task4-config-01.csv ... -13.csv
    table.to_csv(output_folder / f"task4-config-{number:02d}.csv", index=False)

    # Top 20 words other than "time" itself (and "times" without lemmas),
    # and how many of them are in the list of time words.
    top = [w for w in table["word"] if w not in {"time", "times"}][:20]
    summary.append({
        "#": number,
        "configuration": name,
        "time-related in top 20": sum(w in time_words for w in top),
        "top 10": ", ".join(top[:10]),
    })

summary = pd.DataFrame(summary)
# display() is built into Jupyter and shows a formatted table.
with pd.option_context("display.max_colwidth", None):
    display(summary)

'time' is a noun: True | nouns in the vocabulary: 15557


,#,configuration,time-related in top 20,top 10
0,1,BM25_norm * IDF + cosine (task 2),2,"say, one, go, get, come, would, make, people, could, day"
1,2,Relative frequency + cosine (task 3),1,"get, come, go, one, could, work, still, make, say, much"
2,3,BM25_norm * IDF + dot product,0,"cantata, toysrus, flout, routan, hogtied, warpspeed, strategize, collaborating, timeyou, flinch"
3,4,BM25_norm * IDF + Euclidean,2,"say, one, get, go, would, come, make, people, could, like"
4,5,Relative frequency + dot product,0,"girlfriendboyfriend, renewed, asomugha, daytripper, hairdresser, industrialsize, impart, ninetynine, highlighter, chenier"
5,6,Relative frequency + Euclidean,1,"come, one, get, work, could, make, still, say, go, give"
6,7,No lemmatization: BM25_norm * IDF + cosine,0,"said, one, nt, would, people, get, could, like, says, going"
7,8,"BM25_norm * IDF + cosine, frequency >= 20",2,"say, one, go, get, come, would, make, people, could, day"
8,9,"BM25_norm * IDF + cosine, frequency >= 20, nouns",4,"people, day, year, work, way, thing, life, lot, month, week"
9,10,"Relative frequency + cosine, frequency >= 20, nouns",1,"work, day, way, people, thing, job, life, place, part, lot"


### 4.6 Best result

The best configuration is **#12: window 2, $BM25_{norm} \odot IDF$ +
cosine, words seen at least 20 times, nouns only**. 6 of its top 20 words are
time-related: *day*, *year*, *month*, *week*, *season*, *decade*. The next words
(*way*, *thing*, *place*) are not time words, but they are nouns used in the
same phrases (*the first time*, *the first day*, *the right way*).

What each change does:

- **Weighting.** $BM25_{norm} \odot IDF$ beats relative frequency in every
  comparable pair (#1 vs. #2, #9 vs. #10). Relative frequency does not
  saturate high counts or reduce common words, so frequent verbs (*come*,
  *get*, *go*) dominate. BM25 mostly moves the time words up: *year* is
  12th with BM25 vs. 150th with relative frequency, *month* 52nd vs. 239th,
  *week* 58th vs. 443rd. *day* is near the top with both.
- **Measure.** Cosine is the right measure here. The **dot product** (#3,
  #5) ranks rare words first: a word seen once has a short vector with large
  values, and the dot product does not divide by the length. *time* is not
  even first in its own ranking. With the frequency filter (#13) the dot
  product still fails: *time* is only 3rd in its own ranking, and the top
  words are names (*bammel*, *andrus*) whose contexts are almost only *say*.
  **Euclidean distance** (#4, #6) gives a list close to cosine, but not the
  same (*day* is 11th with cosine and 13th with Euclidean). The orders would
  be identical only if every vector had length 1 (L2 norm); ours sum to 1
  (L1), which is a different normalization.
- **Lemmatization.** Without it (#7), *said*, *says*, and *say* are three
  rows, and no time word reaches the top 20. Lemmatization helps.
- **Window.** A window of 2 (#11, #12) keeps the words that fill the same
  slot as *time*; a window of 8 brings in topic words from the whole
  sentence.
- **Nouns only.** The window-8 lists are full of frequent verbs (*say*,
  *go*, *get*). Ranking only nouns (#9, #12) removes them.
- **Frequency >= 20** removes rare words that match by chance. It changes
  nothing for BM25 + cosine (#1 vs. #8), and it is not enough to fix the
  dot product (#3 vs. #13).

In [15]:
# Table of the best configuration: every word that passes its filters,
# from most to least similar (the complete file is in results/).
best_number = 12
best_table = pd.read_csv(output_folder / f"task4-config-{best_number:02d}.csv")
print(configurations[best_number - 1][0])
print("Rows:", len(best_table))
best_table.head(25)

Window 2: BM25_norm * IDF + cosine, frequency >= 20, nouns
Rows: 2087


,word,frequency,cosine similarity
0,time,1955,1.000000
1,day,1393,0.535550
2,year,3476,0.518051
3,way,1063,0.461511
4,people,1979,0.457130
5,thing,823,0.454021
6,work,1465,0.435935
7,place,679,0.426722
8,month,707,0.426354
9,week,899,0.404548


## Conclusion

- **Task 1:** the text was lowercased, tokenized, cleaned (letters only),
  stopwords and COCA markup were removed, and it was lemmatized with spaCy
  (removing stopwords again after lemmatization): 615,788 tokens and 43,313
  vocabulary words.
- **Task 2 ($BM25_{norm} \odot IDF$ + cosine)** and **task 3 (relative
  frequency + cosine)** give mostly frequent verbs (*say*, *go*, *come*)
  with the window of 8. BM25 with IDF moves *year*, *month*, and *week*
  much higher than relative frequency; *day* is near the top with both.
- **Task 4:** the best result comes from BM25 with IDF and cosine, a window
  of 2, and ranking only frequent nouns: *day*, *year*, *month*, *week*,
  *season*, and *decade* are in its top 20.
- Raw counts are biased. BM25 saturates high counts and normalizes context
  length, IDF lowers common context words, and cosine removes the effect of
  vector length. The dot product does not, so it favors words with short,
  concentrated contexts, even with a frequency filter